In [2]:
#Imports
import sys, random
from pathlib import Path

# Walk up from the working directory until we find the project root
cwd = Path.cwd()
for candidate in (cwd,) + tuple(cwd.parents):
    if (candidate / "convoy_sim").is_dir():
        sys.path.append(str(candidate))
        break
else:
    raise RuntimeError("Unable to locate convoy_sim package root")

sys.executable 

'/opt/homebrew/Caskroom/miniforge/base/envs/Python-DS/bin/python'

In [3]:
from convoy_sim.geometry import (
    Point2D,
    Vector2D,
    Vec2,
    as_vec,
    bearing_between,
    closest_approach_time,
    distance,
    min_distance_over_interval,
    rotate_point,
    step_position,
    translate_point,
)
from convoy_sim.entities import Convoy, Ship, Torpedo, torpedo_hits_ship
from convoy_sim.layouts import (
    apply_jitter,
    make_hexagonal_convoy,
    make_rectangular_convoy,
    make_staggered_convoy,
)
from convoy_sim.simulation import (
    run_monte_carlo_attack,
    sample_parallel_torpedoes,
    sample_torpedo_spread_fixed_origin,
    simulate_attack,
    simulate_attack_once,
)

In [6]:

from convoy_sim import (
    as_vec,
    make_rectangular_convoy,
    sample_torpedo_spread_fixed_origin,
    run_monte_carlo_attack,)

layout = dict(
    n_rows=1, n_cols=3,
    spacing_along=400, spacing_across=200,
    speed=5, heading_rad=0,
    length=120, beam=20,
    origin=as_vec(0, 0),)

sampler = lambda rng: sample_torpedo_spread_fixed_origin(
    rng,
    origin=as_vec(-1500, 0),
    speed=25,
    heading_center_rad=0,
    spread_deg=20,
    count=3,
    max_run_time=600,)

result = run_monte_carlo_attack(
    make_rectangular_convoy,
    layout,
    sampler,
    n_trials=10,
    t_max=200.0,)

result["hits_per_trial"]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [7]:
result.keys()
result["expected_hits"]
result["hit_prob_at_least_one"]

1.0

In [8]:
import numpy as np
from convoy_sim import simulate_attack_once

ships = make_rectangular_convoy(
    n_rows=1, n_cols=1,
    spacing_along=100, spacing_across=100,
    speed=0, heading_rad=0,
    length=120, beam=20,
    origin=as_vec(0, 0),
)

torps = sample_torpedo_spread_fixed_origin(
    np.random.default_rng(0),
    origin=as_vec(-1000, 0),
    speed=20,
    heading_center_rad=0,
    spread_deg=0,
    count=1,
    max_run_time=200,
)

simulate_attack_once(ships, torps, t_max=120)

1

In [9]:
from convoy_sim import make_hexagonal_convoy

ships = make_hexagonal_convoy(
    n_rows=2, n_cols=3,
    spacing_along=400, spacing_across=200,
    speed=5, heading_rad=0,
    length=120, beam=20,
    origin=as_vec(0, 0),
)

[ship.position for ship in ships]

[array([-200., -200.]),
 array([0., 0.]),
 array([-200.,  200.]),
 array([ 200., -100.]),
 array([400., 100.]),
 array([200., 300.])]

In [10]:
for i, s in enumerate(ships):
    print(i, s.position)

0 [-200. -200.]
1 [0. 0.]
2 [-200.  200.]
3 [ 200. -100.]
4 [400. 100.]
5 [200. 300.]


In [5]:
from scenarios.scenario_a1_constraints import build_scenario_a1
from convoy_sim.simulation import run_monte_carlo_attack

scenario = build_scenario_a1(n_trials=1000, rng_seed=0)
result = run_monte_carlo_attack(
    layout_fn=scenario.layout_fn,
    layout_kwargs=scenario.layout_kwargs,
    torpedo_sampler=scenario.torpedo_sampler,
    n_trials=scenario.n_trials,
    t_max=scenario.t_max,
    rng=None,
    max_hits_per_torpedo=1,
)
print(result["expected_hits"], result["var_hits"], result["hit_prob_at_least_one"])


ValueError: Unable to generate feasible attack proposal: {'failed_checks': ['range', 'approach_mode'], 'range_m': 4045.975060186077, 'convoy_reference': {'centroid': array([0., 0.]), 'heading_rad': 0.0, 'speed': 5.0, 'bbox_along': 1200.0, 'bbox_across': 1050.0}, 'detection_risk': 1.8939772944962352, 'env': Environment(time_of_day='day', visibility_m=6000.0, sea_state=3, detection_risk_scale=1.0)}